# Jamii Afya Phase 03 — Qwen3-0.6B-Base QLoRA training (GPU)

Runs the repo's locked training recipe on Kaggle GPUs instead of RunPod:
listwise MCQ ranking + clinical chat SFT + Track-B healthcare corpus, one epoch
with step checkpoints, then merge + domain-imatrix Q4_0 export.

One epoch keeps the run inside Kaggle's 9h session limit; step checkpoints
(`--save_steps 500`) let a follow-up run continue via `--resume_adapter`
if the session ends early. Artifacts land in `/kaggle/working/phase03-results/`.

Recipe source: `scripts/train_lora.py` + `scripts/export_gguf.sh` on branch
`research/phase-0-1-baseline-evals`. No recipe changes here — only paths and
epoch/checkpoint settings for the Kaggle time budget.

In [ ]:
import subprocess
import sys
from pathlib import Path

WORK = Path('/kaggle/working')
REPO = WORK / 'adtc-llm-limited-hardware'
OUT = WORK / 'phase03-results'
BRANCH = 'research/phase-0-1-baseline-evals'

def run(command, cwd=None, log=None):
    print('+', ' '.join(str(c) for c in command), flush=True)
    r = subprocess.run([str(c) for c in command], cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False)
    print(r.stdout[-3000:], flush=True)
    if log:
        Path(log).parent.mkdir(parents=True, exist_ok=True)
        Path(log).write_text(r.stdout, encoding='utf-8')
    if r.returncode:
        raise RuntimeError(f'exit {r.returncode}: {command}')
    return r

OUT.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
         'https://github.com/qeinstein/adtc-llm-limited-hardware.git', str(REPO)])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-dev.txt'],
    cwd=REPO, log=OUT / 'pip_install.log')
run([sys.executable, '-c', 'import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)'])
print('repo sha:', run(['git', 'rev-parse', 'HEAD'], cwd=REPO).stdout.strip())

In [ ]:
# Track A (MCQA) + Track B (healthcare corpus) + splits/calibration corpus.
# Caps match the PROGRESS.md RunPod recipe.
run([sys.executable, 'scripts/build_accuracy_sft.py', '--max-per-dataset', '20000'],
    cwd=REPO, log=OUT / 'build_accuracy_sft.log')
run([sys.executable, 'scripts/build_healthcare_corpus.py', '--max-per-dataset', '15000'],
    cwd=REPO, log=OUT / 'build_healthcare_corpus.log')
run([sys.executable, 'scripts/prepare_dataset.py'], cwd=REPO, log=OUT / 'prepare_dataset.log')

In [ ]:
# One epoch, batch 4 / accum 8 / max_len 256 / grad-checkpointing on:
# the exact config that survived OOM debugging (see PROGRESS.md).
# Step checkpoints allow --resume_adapter continuation in a second session.
run([sys.executable, 'scripts/train_lora.py', '--base_model', 'Qwen/Qwen3-0.6B-Base',
     '--epochs', '1', '--batch_size', '4', '--grad_accum', '8',
     '--max_len', '256', '--save_steps', '500',
     '--output_dir', str(REPO / 'output' / 'jamii-lora')],
    cwd=REPO, log=OUT / 'train.log')

In [ ]:
import os
# Native llama.cpp build for the convert/quantize path (export script default).
env = dict(os.environ, QUANT='Q4_0')
print('+ QUANT=Q4_0 bash scripts/export_gguf.sh', flush=True)
r = subprocess.run(['bash', 'scripts/export_gguf.sh'], cwd=REPO, text=True,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     env={**env, 'PYTHON': sys.executable}, check=False)
print(r.stdout[-3000:], flush=True)
(OUT / 'export.log').write_text(r.stdout, encoding='utf-8')
if r.returncode:
    raise RuntimeError(f'export exit {r.returncode}')
for g in sorted((REPO / 'model').glob('*.gguf')):
    print(f'{g.name}: {g.stat().st_size / 1e6:.1f} MB', flush=True)

In [ ]:
# Post-train probe: same MCQ harness as Phase 02, limit 50 (CPU-side, quick).
# Compares against the 51-57% arc_easy pre-fine-tune baseline in PROGRESS.md.
run([sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python==0.3.16'],
    log=OUT / 'pip_llama.log')
for gguf in sorted((REPO / 'model').glob('*.gguf')):
    for task in ('arc_easy', 'medmcqa'):
        run([sys.executable, 'scripts/mcq_eval.py', '--model', str(gguf),
             '--task', task, '--limit', '50'], cwd=REPO,
            log=OUT / f'eval_{gguf.stem}_{task}.log')
print('copy final artifacts to output dir')
run(['cp', '-v'] + [str(g) for g in sorted((REPO / 'model').glob('*.gguf'))] + [str(OUT)])
run(['cp', '-rv', str(REPO / 'output' / 'jamii-lora'), str(OUT / 'jamii-lora')])
print('DONE — see phase03-results/')